In [ ]:
!pip install workalendar -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 138.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 6.7 MB/s eta 0:00:00


In [ ]:
!pip install kaleido==0.2.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 28.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit
from workalendar.europe import Russia
from tqdm import tqdm

In [ ]:
df = pd.read_csv("df_final+whether.csv", parse_dates=["timestamp"])
houses = [col for col in df.columns if col.startswith("house_")]

cal = Russia()
def is_holiday(dt):
    if dt.weekday() >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df["is_holiday"] = df["timestamp"].apply(is_holiday)

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"MAE": round(mae, 3), "RMSE": round(rmse, 3), "MAPE": round(mape, 3)}

horizons = {
    "4h":  8,
    "8h":  16,
    "24h": 48,
    "7d":  336,
    "14d": 672,
    "1m":  1488,
}

tscv = TimeSeriesSplit(n_splits=5)

print(f"Строк: {len(df)}")
print(f"Дома: {houses}")

Строк: 36260
Дома: ['house_1', 'house_2', 'house_3', 'house_4', 'house_5', 'house_6', 'house_7', 'house_8']


In [ ]:
def make_features(df, house, lag_features=[1, 2, 48, 96, 336]):
    data = df[["timestamp", house]].copy()

    # временные признаки
    data["hour"] = data["timestamp"].dt.hour
    data["minute"] = data["timestamp"].dt.minute
    data["weekday"] = data["timestamp"].dt.weekday
    data["month"] = data["timestamp"].dt.month
    data["day_of_year"] = data["timestamp"].dt.dayofyear
    data["is_weekend"] = (data["weekday"] >= 5).astype(int)
    data["is_holiday"] = df["is_holiday"].values

    # лаговые признаки
    for lag in lag_features:
        data[f"lag_{lag}"] = data[house].shift(lag)

    # скользящее среднее
    data["rolling_mean_48"] = data[house].shift(1).rolling(48).mean()
    data["rolling_mean_336"] = data[house].shift(1).rolling(336).mean()

    # погодные признаки
    data["temp_c"] = df["temp_c"]
    data["humidity"] = df["humidity"]
    data["cloudiness"] = df["cloudiness"]

    data = data.dropna().reset_index(drop=True)
    feature_cols = [c for c in data.columns if c not in ["timestamp", house]]
    X = data[feature_cols]
    y = data[house]

    return X, y, data["timestamp"]

# проверка
X, y, ts = make_features(df, "house_1")
print(f"Признаков: {X.shape[1]}")
print(f"Строк: {X.shape[0]}")
print(f"Признаки: {X.columns.tolist()}")

Признаков: 17
Строк: 35924
Признаки: ['hour', 'minute', 'weekday', 'month', 'day_of_year', 'is_weekend', 'is_holiday', 'lag_1', 'lag_2', 'lag_48', 'lag_96', 'lag_336', 'rolling_mean_48', 'rolling_mean_336', 'temp_c', 'humidity', 'cloudiness']


In [ ]:
xgb_results = {}

for house in houses:
    house_results = {}
    X_full, y_full, ts_full = make_features(df, house)

    for horizon_name, horizon_points in tqdm(
        horizons.items(),
        desc=f"XGBoost {house}",
        total=len(horizons)
    ):
        fold_metrics = []

        for fold, (train_idx, val_idx) in enumerate(tscv.split(X_full)):
            X_train = X_full.iloc[train_idx]
            y_train = y_full.iloc[train_idx]
            X_val = X_full.iloc[val_idx[:horizon_points]]
            y_val = y_full.iloc[val_idx[:horizon_points]]

            if len(y_val) < horizon_points:
                continue

            model = XGBRegressor(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=-1,
                verbosity=0
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            metrics = evaluate(y_val.values, y_pred)
            fold_metrics.append(metrics)

        house_results[horizon_name] = {
            "MAE":  round(np.mean([m["MAE"]  for m in fold_metrics]), 3),
            "RMSE": round(np.mean([m["RMSE"] for m in fold_metrics]), 3),
            "MAPE": round(np.mean([m["MAPE"] for m in fold_metrics]), 3),
        }

    xgb_results[house] = house_results

# таблицы
rows = []
for house in houses:
    for horizon_name, metrics in xgb_results[house].items():
        rows.append({
            "модель": "XGBoost",
            "дом": house,
            "горизонт": horizon_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "MAPE": metrics["MAPE"],
        })

df_xgb = pd.DataFrame(rows)

for metric in ["MAE", "RMSE", "MAPE"]:
    pivot = df_xgb.pivot(index="горизонт", columns="дом", values=metric)
    pivot = pivot.reindex(list(horizons.keys()))
    print(f"\nXGBoost - {metric}:")
    print(pivot.to_string())

df_xgb.to_csv("results_xgboost.csv", index=False)

XGBoost house_8: 100%|██████████| 6/6 [00:24<00:00,  4.04s/it]


XGBoost - MAE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
горизонт                                                                        
4h          1.771    1.090    1.067    1.310    1.266    2.468    2.568    1.155
8h          1.683    1.061    0.931    1.282    1.228    2.376    2.350    1.241
24h         1.636    1.089    0.854    1.295    1.190    2.218    2.218    1.204
7d          1.858    1.176    0.919    1.415    1.306    2.298    2.315    1.266
14d         1.857    1.156    0.922    1.423    1.313    2.307    2.408    1.266
1m          1.933    1.225    0.919    1.450    1.339    2.457    2.507    1.324

XGBoost - RMSE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
горизонт                                                                        
4h          2.240    1.410    1.334    1.616    1.488    2.969    3.098    1.468
8h          2.128    1.372    1.182    1.671    1.494    2.963    3.092    1

In [ ]:
lgbm_results = {}

for house in houses:
    house_results = {}
    X_full, y_full, ts_full = make_features(df, house)

    for horizon_name, horizon_points in tqdm(
        horizons.items(),
        desc=f"LightGBM {house}",
        total=len(horizons)
    ):
        fold_metrics = []

        for fold, (train_idx, val_idx) in enumerate(tscv.split(X_full)):
            X_train = X_full.iloc[train_idx]
            y_train = y_full.iloc[train_idx]
            X_val = X_full.iloc[val_idx[:horizon_points]]
            y_val = y_full.iloc[val_idx[:horizon_points]]

            if len(y_val) < horizon_points:
                continue

            model = LGBMRegressor(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=-1,
                verbose=-1
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            metrics = evaluate(y_val.values, y_pred)
            fold_metrics.append(metrics)

        house_results[horizon_name] = {
            "MAE":  round(np.mean([m["MAE"]  for m in fold_metrics]), 3),
            "RMSE": round(np.mean([m["RMSE"] for m in fold_metrics]), 3),
            "MAPE": round(np.mean([m["MAPE"] for m in fold_metrics]), 3),
        }

    lgbm_results[house] = house_results

rows = []
for house in houses:
    for horizon_name, metrics in lgbm_results[house].items():
        rows.append({
            "модель": "LightGBM",
            "дом": house,
            "горизонт": horizon_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "MAPE": metrics["MAPE"],
        })

df_lgbm = pd.DataFrame(rows)

for metric in ["MAE", "RMSE", "MAPE"]:
    pivot = df_lgbm.pivot(index="горизонт", columns="дом", values=metric)
    pivot = pivot.reindex(list(horizons.keys()))
    print(f"\nLightGBM - {metric}:")
    print(pivot.to_string())

df_lgbm.to_csv("results_lgbm.csv", index=False)

LightGBM house_8: 100%|██████████| 6/6 [00:15<00:00,  2.61s/it]


LightGBM - MAE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
горизонт                                                                        
4h          1.962    1.068    1.004    1.331    1.242    2.508    2.544    1.223
8h          1.790    1.037    0.909    1.336    1.178    2.408    2.377    1.260
24h         1.671    1.104    0.845    1.306    1.163    2.240    2.243    1.198
7d          1.866    1.168    0.899    1.403    1.268    2.276    2.337    1.237
14d         1.866    1.141    0.900    1.413    1.273    2.315    2.419    1.229
1m          1.894    1.144    0.906    1.433    1.269    2.418    2.599    1.274

LightGBM - RMSE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
горизонт                                                                        
4h          2.407    1.427    1.287    1.607    1.448    3.021    3.152    1.566
8h          2.282    1.394    1.160    1.719    1.433    2.968    3.173   

In [ ]:
# графики для XGBoost
train_idx, val_idx = list(tscv.split(X_full))[-1]
house = "house_1"
X_full, y_full, ts_full = make_features(df, house)

X_train = X_full.iloc[train_idx]
y_train = y_full.iloc[train_idx]

model_xgb = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, verbosity=0
)
model_xgb.fit(X_train, y_train)

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    X_val = X_full.iloc[val_idx[:horizon_points]]
    y_val = y_full.iloc[val_idx[:horizon_points]]
    timestamps = ts_full.iloc[val_idx[:horizon_points]]
    y_pred = model_xgb.predict(X_val)

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_val.values,
        mode="lines", name="факт",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_pred,
        mode="lines", name="прогноз XGBoost",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.update_xaxes(title_text="Дата", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="кВт", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="XGBoost: прогнозные и фактические значения по всем горизонтам прогнозирования (дом 1, fold 5)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image("30_xgb_forecast_all_horizons.png", scale=2)

In [ ]:
model_lgbm = LGBMRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, verbose=-1
)
model_lgbm.fit(X_train, y_train)

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    X_val = X_full.iloc[val_idx[:horizon_points]]
    y_val = y_full.iloc[val_idx[:horizon_points]]
    timestamps = ts_full.iloc[val_idx[:horizon_points]]
    y_pred = model_lgbm.predict(X_val)

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_val.values,
        mode="lines", name="факт",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_pred,
        mode="lines", name="прогноз LightGBM",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.update_xaxes(title_text="Дата", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="кВт", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="LightGBM: Прогнозные и фактические ззначения по всем горизонтам прогнозирования (дом 1, fold 5)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image("31_lgbm_forecast_all_horizons.png", scale=2)

In [ ]:
# важность признаков XGBoost
feature_names = X_full.columns.tolist()
importances_xgb = model_xgb.feature_importances_
importances_lgbm = model_lgbm.feature_importances_

# сортировка по важности
idx_xgb = np.argsort(importances_xgb)[::-1]
idx_lgbm = np.argsort(importances_lgbm)[::-1]

fig = make_subplots(rows=1, cols=2, subplot_titles=("XGBoost", "LightGBM"))

fig.add_trace(go.Bar(
    x=[feature_names[i] for i in idx_xgb],
    y=importances_xgb[idx_xgb],
    marker_color="#d62728",
    showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=[feature_names[i] for i in idx_lgbm],
    y=importances_lgbm[idx_lgbm],
    marker_color="#d62728",
    showlegend=False
), row=1, col=2)

fig.update_layout(
    title="Важность признаков (дом 1)",
    width=700, height=400,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=80),
)

fig.update_xaxes(tickangle=45)
fig.show()
fig.write_image("32_feature_importance.png", scale=2)

XGBoost:

lag_336 - самый важный признак (0.65!) - потребление неделю назад
lag_1 - второй по важности (0.21) - потребление 30 минут назад
lag_48 - третий - потребление вчера в то же время
остальные признаки почти нулевые

LightGBM:

более равномерное распределение важности
lag_1 и hour на первых местах
lag_336, lag_48, lag_2 тоже значимы
погодные признаки (temp_c, humidity) в середине